# DS 6021 — Hierarchical Clustering
## Interactive lecture companion · Intro to Predictive Modeling

**Learning goals:** Understand what distances and linkage rules do, follow a dendrogram, compare cuts using silhouette scores, connect Ward's method to K-means, and apply Gower dissimilarity to mixed data.

We use **synthetic viewer preferences** for *Breaking Bad* and *Love Is Blind* to connect this lesson to our previous K-means discussion. All ratings are fictional and are used only for instruction.

**Main idea:** Start with one viewer per cluster → merge closest groups → build dendrogram → decide where to cut. Unlike K-means, **we do not choose K before building the tree**.

### 1. Begin with four viewers: calculate the first distances ourselves

We have four people rating the same two shows from 1 to 5. Can you guess who merges first?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from sklearn.metrics import silhouette_score, silhouette_samples

viewers4 = pd.DataFrame({
    "Breaking Bad": [1, 1, 4, 4],
    "Love Is Blind": [1, 2, 4, 5],
}, index=["A", "B", "C", "D"])

viewers4

In [ ]:
ax = viewers4.plot.scatter(x="Breaking Bad", y="Love Is Blind", s=110, figsize=(6, 5))
for name, row in viewers4.iterrows():
    ax.annotate(name, (row["Breaking Bad"] + 0.07, row["Love Is Blind"] + 0.07))
ax.set(xlim=(0.5, 5.5), ylim=(0.5, 5.5), title="Four viewers — which ones are closest?")
ax.grid(alpha=0.2)
plt.show()

In [ ]:
distances4 = pd.DataFrame(
    squareform(pdist(viewers4, metric="euclidean")),
    index=viewers4.index,
    columns=viewers4.index
)
distances4.round(2)

**Think:** Distance between A and B is 1; between C and D is 1. These are the first two merges (their order is a tie). But after the merges, **what is the distance between the groups {A,B} and {C,D}?** We now need a linkage rule.

### 2. Linkage means distance *between groups*

Four cross-cluster distances are involved. Compare **single** (smallest), **complete** (largest), and **average** (mean).

In [ ]:
cross_distances = distances4.loc[["A", "B"], ["C", "D"]]
print("Distances connecting the two groups:")
print(cross_distances.round(2))
print("Single linkage:", round(cross_distances.values.min(), 2))
print("Complete linkage:", round(cross_distances.values.max(), 2))
print("Average linkage:", round(cross_distances.values.mean(), 2))

In [ ]:
# Use one method at a time, just as in the dashboard.
method = "single"   # change to "complete", "average", or "ward"
Z4 = linkage(viewers4, method=method, metric="euclidean")

plt.figure(figsize=(8, 4))
dendrogram(Z4, labels=viewers4.index.tolist())
plt.ylabel("Linkage height")
plt.title(f"Dendrogram — {method} linkage")
plt.show()

# The linkage matrix records each merge and the height of that merge.
pd.DataFrame(Z4, columns=["Group 1", "Group 2", "Height", "New size"]).round(2)

**Read the dendrogram:** Each leaf is one viewer. A horizontal branch means two groups merged. Its **height is the chosen linkage quantity** (Ward is an exception: SciPy uses the square root of twice the SSE increase).

**Question:** If the data do not change, why does the height of the final merge change when you switch from single to complete linkage?

### 3. Cut the tree: choose K *after* fitting

In [ ]:
Z4 = linkage(viewers4, method="average")
clusters4 = fcluster(Z4, t=2, criterion="maxclust")

result4 = viewers4.copy()
result4["cluster"] = clusters4
result4

Changing `t=2` to a different number of groups changes the **cut**, not the hierarchy. With just four observations, different choices may be less informative. Next, we use a slightly larger example.

### 4. A larger set of fictional viewers

This matches the main dashboard's default dataset. Two numerical variables are on the same 1–5 rating scale; standardization can still be considered, but it is not automatically required.

In [ ]:
rating_pairs = [
    (1.0,4.6),(1.4,4.8),(1.6,4.3),(1.2,3.9),
    (4.4,1.0),(4.8,1.3),(4.3,1.6),(3.9,1.2),
    (3.4,4.3),(3.7,3.9),(4.1,4.7),(3.2,4.7),
    (2.2,3.2),(2.8,2.7)
]
habits = ["Binge","Binge","Weekly","Binge", "Weekly","Weekly","Binge","Weekly",
          "Binge","Weekly","Binge","Weekly", "Binge","Weekly"]

viewers = pd.DataFrame(rating_pairs, columns=["Breaking Bad", "Love Is Blind"])
viewers.index = [f"V{i+1:02d}" for i in range(len(viewers))]
viewers["Viewing habit"] = habits
viewers.head()

In [ ]:
X = viewers[["Breaking Bad", "Love Is Blind"]]
Z = linkage(X, method="average", metric="euclidean")

plt.figure(figsize=(11, 4))
dendrogram(Z, labels=viewers.index.tolist(), color_threshold=0)
plt.ylabel("Average-linkage distance")
plt.title("Streaming-viewer hierarchy")
plt.xticks(rotation=45)
plt.show()

In [ ]:
K = 3
viewers["cluster"] = fcluster(Z, t=K, criterion="maxclust")

fig, ax = plt.subplots(figsize=(7, 5))
for group, part in viewers.groupby("cluster"):
    ax.scatter(part["Breaking Bad"], part["Love Is Blind"], s=80, label=f"Cluster {group}")
    for name, row in part.iterrows():
        ax.annotate(name, (row["Breaking Bad"]+0.04, row["Love Is Blind"]+0.05), fontsize=8)
ax.set(xlabel="Breaking Bad rating", ylabel="Love Is Blind rating",
       title=f"Average linkage, K = {K}", xlim=(0.5,5.5), ylim=(0.5,5.5))
ax.legend()
plt.show()

**Interpret:** Describe the preferences of each cluster. What happens to the group memberships when you change `K=3` to `K=4`? Try `method="single"` and `method="complete"` and ask which groups seem coherent.

### 5. Which cut has the best average silhouette?

Recall our K-means lesson: for each viewer, silhouette compares the mean distance to their own group, $a(i)$, with the mean distance to the **nearest other group**, $b(i)$:

$$s(i)=\frac{b(i)-a(i)}{\max\{a(i),b(i)\}}.$$

In [ ]:
results = []
for k in range(2, 10):
    labels = fcluster(Z, t=k, criterion="maxclust")
    # Avoid invalid silhouette calculations if a tie gives fewer than k groups.
    if 1 < len(np.unique(labels)) < len(X):
        avg = silhouette_score(X, labels, metric="euclidean")
        results.append((k, avg))

silhouette_table = pd.DataFrame(results, columns=["K", "Average silhouette"])
silhouette_table.round(3)

In [ ]:
ax = silhouette_table.plot(x="K", y="Average silhouette", marker="o", legend=False, figsize=(7, 4))
ax.set(title="Average silhouette vs. K", ylabel="Average silhouette")
ax.grid(alpha=0.2)
plt.show()

best_k = int(silhouette_table.loc[silhouette_table["Average silhouette"].idxmax(), "K"])
print("Highest average silhouette at K =", best_k)

In [ ]:
K = 3
labels = fcluster(Z, t=K, criterion="maxclust")
individual_scores = silhouette_samples(X, labels, metric="euclidean")
profile = pd.DataFrame({"Viewer": X.index, "Cluster": labels, "Silhouette": individual_scores})
profile = profile.sort_values(["Cluster", "Silhouette"])

fig, ax = plt.subplots(figsize=(9, 5))
for group, subset in profile.groupby("Cluster"):
    ax.barh(subset["Viewer"], subset["Silhouette"], label=f"Cluster {group}")
ax.axvline(0, color="black", lw=1)
ax.set(xlabel="Individual silhouette score", ylabel="Viewer", title=f"Silhouette profiles for K = {K}")
ax.legend()
plt.show()

**Interpret:** An average is not the full story. Which viewers are on cluster boundaries or have negative silhouettes? If `K=4` has the highest average, must we choose 4? Explain how subject-matter interpretation and stability could matter.

### 6. Ward: hierarchical clustering with an SSE objective

For groups $A$ and $B$ with sizes $n_A,n_B$ and mean vectors $\bar{x}_A,\bar{x}_B$:

$$\Delta\mathrm{SSE}=\frac{n_A n_B}{n_A+n_B}\|\bar x_A-\bar x_B\|^2.$$

This is why Ward resembles K-means. K-means reassigns points for a chosen K; Ward greedily merges groups to minimize the *increase* in SSE.

In [ ]:
A = viewers4.loc[["A", "B"]].to_numpy()
B = viewers4.loc[["C", "D"]].to_numpy()

increase_in_sse = (len(A)*len(B)/(len(A)+len(B))) * np.sum((A.mean(axis=0)-B.mean(axis=0))**2)
print("Ward SSE increase when joining these two groups:", increase_in_sse)
print("Corresponding SciPy Ward dendrogram height:", np.sqrt(2*increase_in_sse))

Z_ward = linkage(X, method="ward")
plt.figure(figsize=(10, 4))
dendrogram(Z_ward, labels=viewers.index.tolist())
plt.title("Ward hierarchical clustering")
plt.ylabel("Ward height = sqrt(2 × SSE increase)")
plt.show()

### 7. Gower: numeric + categorical features

The two axes on the scatterplot cannot show the categorical feature. Gower averages (1) normalized numeric differences and (2) 0/1 categorical mismatches. For this small complete dataset, we can compute it in a transparent loop rather than hiding the logic in a library.

**Important:** Do **not** use Ward on the resulting Gower dissimilarity matrix; use single, complete, or average linkage.

In [ ]:
n = len(viewers)
D_gower = np.zeros((n, n))

x_range = viewers["Breaking Bad"].max() - viewers["Breaking Bad"].min()
y_range = viewers["Love Is Blind"].max() - viewers["Love Is Blind"].min()

for i in range(n):
    for j in range(n):
        d1 = abs(viewers.iloc[i]["Breaking Bad"] - viewers.iloc[j]["Breaking Bad"]) / x_range
        d2 = abs(viewers.iloc[i]["Love Is Blind"] - viewers.iloc[j]["Love Is Blind"]) / y_range
        d3 = 0 if viewers.iloc[i]["Viewing habit"] == viewers.iloc[j]["Viewing habit"] else 1
        D_gower[i, j] = (d1 + d2 + d3) / 3

pd.DataFrame(D_gower, index=viewers.index, columns=viewers.index).round(2).iloc[:6, :6]

In [ ]:
Z_gower = linkage(squareform(D_gower), method="average")

plt.figure(figsize=(11, 4))
dendrogram(Z_gower, labels=viewers.index.tolist())
plt.ylabel("Average Gower dissimilarity")
plt.title("Gower + average linkage")
plt.show()

labels_gower = fcluster(Z_gower, t=3, criterion="maxclust")
print("Average silhouette using Gower:",
      round(silhouette_score(D_gower, labels_gower, metric="precomputed"), 3))

**Reflection questions**

1. Compare numeric Euclidean + average linkage with Gower + average linkage. Why might a viewer switch groups?
2. Would it be fair to give a categorical variable the same weight as two numerical features? What judgment are we making?
3. Why is it not meaningful to calculate a centroid of a nominal category such as *Binge* / *Weekly*?
4. What rule would you need if a brand-new viewer arrives and wants a cluster assignment? (A dendrogram by itself does not prescribe one.)

**Suggested reading:** *Introduction to Statistical Learning with Applications in Python* (chapter 12, hierarchical clustering); optional: *Elements of Statistical Learning* on hierarchical clustering.